#### 1. Loading the data
The classified CISA events and discussion questions are loaded to create a dataset of cybersecurity exercises.

In [7]:
import pandas as pd
from pathlib import Path

data_dir = Path("../data/processed")

events = pd.read_csv(data_dir / "cisa_classified_events.csv")
questions = pd.read_csv(data_dir / "cisa_discussion_questions.csv")

print("Events:", events.shape)
print("Questions:", questions.shape)

Events: (66, 10)
Questions: (430, 9)


#### 2. Examining exercise structure
The modules and vignettes are compared to see how events and questions are organized across the manuals.

In [8]:
print("Events by section:")
display(
    events.groupby(["manual_title", "section"])
    .size()
    .reset_index(name="event_count")
)

print("Questions by section:")
display(
    questions.groupby(["manual_title", "section"])
    .size()
    .reset_index(name="question_count")
)

Events by section:


,manual_title,section,event_count
0,Communications,Module 1,4
1,Communications,Module 2,4
2,Executives,Vignette 1: Whaling/Ransomware,1
3,Executives,Vignette 2: Insider Threat,1
4,Financial Services,Module 1,4
5,Financial Services,Module 2,4
6,Healthcare,Module 1,5
7,Healthcare,Module 2,6
8,Industrial Control Systems,Module 1,5
9,Industrial Control Systems,Module 2,4


Questions by section:


,manual_title,section,question_count
0,Communications,Module 1,23
1,Communications,Module 2,22
2,Executives,General Discussion Questions,20
3,Executives,Vignette 1: Whaling/Ransomware,12
4,Executives,Vignette 2: Insider Threat,17
5,Financial Services,Module 1,29
6,Financial Services,Module 2,26
7,Healthcare,Module 1,34
8,Healthcare,Module 2,27
9,Industrial Control Systems,Module 1,32


#### 3. Creating the exercise dataset
Events and discussion questions belonging to the same module or vignette are combined into one exercise record.

In [9]:
# combines events belonging to the same exercise
exercise_events = events.groupby(
    ["manual_title", "section"], sort=False
).agg(
    event_count=("inject_id", "size"),
    event_ids=("inject_id", list),
    threat_categories=("threat_category", lambda x: sorted(x.unique())),
    impact_categories=("impact_category", lambda x: sorted(x.unique())),
    event_text=("event_description", lambda x: " ".join(x))
).reset_index()

# combines discussion questions belonging to the same exercise
exercise_questions = questions.groupby(
    ["manual_title", "section"], sort=False
).agg(
    question_count=("question_id", "size"),
    question_ids=("question_id", list),
    question_text=("question_text", lambda x: " ".join(x))
).reset_index()

# matches events and questions by manual and section
exercises = exercise_events.merge(
    exercise_questions,
    on=["manual_title", "section"],
    how="left",
    validate="one_to_one"
)

# adds unique exercise IDs
exercises["exercise_id"] = [
    f"EX-{i:02d}" for i in range(1, len(exercises) + 1)
]

print("Total exercises:", len(exercises))

display(exercises[
    ["exercise_id", "manual_title", "section",
     "event_count", "question_count",
     "threat_categories", "impact_categories"]
])

Total exercises: 16


,exercise_id,manual_title,section,event_count,question_count,threat_categories,impact_categories
0,EX-01,Healthcare,Module 1,5,34,"[No Confirmed Incident, Phishing, System Disru...","[No Direct Impact, Operational Disruption, Sec..."
1,EX-02,Healthcare,Module 2,6,27,"[Data Exposure, Insider Threat / Unauthorized ...","[Data Exposure, Operational Disruption, Patien..."
2,EX-03,Vendor,Module 1,4,28,"[No Confirmed Incident, Third-Party / Supply C...","[Data Exposure, No Direct Impact]"
3,EX-04,Vendor,Module 2 – 6 months later,4,21,"[Other Security Incident, System Disruption, T...","[Operational Disruption, Security Risk]"
4,EX-05,Ransomware,Module 1,4,27,"[No Confirmed Incident, Other Security Inciden...","[No Direct Impact, Security Risk]"
5,EX-06,Ransomware,Module 2,4,32,"[Data Exposure, Ransomware]","[Data Exposure, Operational Disruption, Reputa..."
6,EX-07,Insider Threat,Module 1,5,22,"[Data Exposure, Insider Threat / Unauthorized ...","[Data Exposure, Financial Cost, Personnel Cons..."
7,EX-08,Insider Threat,Module 2,7,31,"[Insider Threat / Unauthorized Access, Third-P...","[Data Loss, Financial Cost, Security Risk]"
8,EX-09,Executives,Vignette 1: Whaling/Ransomware,1,12,[Ransomware],[Operational Disruption]
9,EX-10,Executives,Vignette 2: Insider Threat,1,17,[Insider Threat / Unauthorized Access],[Data Exposure]


#### 4. Checking the exercise dataset
The number of exercises, events, and associated questions is checked against the original data.

In [10]:
print("Exercises:", len(exercises))
print("Events included:", exercises["event_count"].sum())
print("Questions linked:", exercises["question_count"].sum())

# checks that all events are included
assert exercises["event_count"].sum() == len(events)

# checks that all exercise records have questions
assert exercises["question_count"].notna().all()

# inspects the general questions from the Executives manual
general_questions = questions[
    (questions["manual_title"] == "Executives") &
    (questions["section"] == "General Discussion Questions")
]

print("General questions kept separately:", len(general_questions))

Exercises: 16
Events included: 66
Questions linked: 410
General questions kept separately: 20


#### 5. Creating exercise descriptions
The scenario events and discussion questions are combined into one text field for each exercise.

In [12]:
exercises["exercise_text"] = (
    exercises["event_text"].fillna("")
    + " "
    + exercises["question_text"].fillna("")
)

display(exercises[["exercise_id", "manual_title", "section", "exercise_text"]].head())

,exercise_id,manual_title,section,exercise_text
0,EX-01,Healthcare,Module 1,The Cybersecurity and Infrastructure Security ...
1,EX-02,Healthcare,Module 2,A system administrator discovers active accoun...
2,EX-03,Vendor,Module 1,CISA and the Federal Bureau of Investigation (...
3,EX-04,Vendor,Module 2 – 6 months later,Employees at your organization are unable to s...
4,EX-05,Ransomware,Module 1,The Cybersecurity and Infrastructure Security ...


#### 6. Saving the exercise dataset
The structured exercise records are saved as a CSV file for further analysis.

In [13]:
# checks the dataset
assert len(exercises) == 16
assert exercises["exercise_id"].is_unique
assert exercises["exercise_text"].notna().all()

# saves as CSV
exercises.to_csv(
    data_dir / "cisa_exercises.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Exercises saved:", len(exercises))
print("Columns:", list(exercises.columns))

Exercises saved: 16
Columns: ['manual_title', 'section', 'event_count', 'event_ids', 'threat_categories', 'impact_categories', 'event_text', 'question_count', 'question_ids', 'question_text', 'exercise_id', 'exercise_text']
